## 3 Omnilingual Evaluation

This notebook evaluates the finetuned Omnilingual model.

In [ ]:
import os
import sys
import torch
from tqdm import tqdm
from pathlib import Path
from omnilingual_asr.data import load_all_data
from omnilingual_asr.evaluate import add_metrics_columns, idiom_summary, print_evaluation_summary, show_examples
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs

NOTEBOOK_DIR = Path.cwd()                             # scripts/
ROOT_DIR = NOTEBOOK_DIR.parent                        # omnilingual/
SUBMODULE_ROOT = ROOT_DIR / "omnilingual_asr"         # submodule root (contains workflows/)

# We need to add the submodule root to PYTHONPATH so the subprocess can find 'workflows'
env = os.environ.copy()
pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = str(SUBMODULE_ROOT) + (f":{pythonpath}" if pythonpath else "")

# Also add to our own sys.path for the helper imports (already installed, but we need it for early imports)
sys.path.insert(0, str(ROOT_DIR))

from omnilingual_asr.utils import get_best_gpu, normalize_romansh_text

best_gpu = get_best_gpu()
if torch.cuda.is_available():
    os.environ["CUDA_VISIBLE_DEVICES"] = str(best_gpu)
    print(f"Using GPU {best_gpu}")
else:
    print("No GPU available – falling back to CPU")


Selected GPU 5 with 24121 MiB free memory
Using GPU 5


In [23]:
CHECKPOINT_FILE = "/local/scratch/matuor/omnilingual/models/omnilingual-ctc-rm-1b/ws_1.8f5bfbc3/checkpoints/step_5000/model/pp_00/tp_00/sdp_00.pt"
LANGUAGE_CODE = "roh_Latn_surs1244"
BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}, Lang supported: {LANGUAGE_CODE in supported_langs}")

# 1. Initialize the inference pipeline using the BASE model card. 
base_model_card = "omniASR_CTC_1B"
print("Loading base model architecture...")
pipeline = ASRInferencePipeline(model_card=base_model_card, device=DEVICE)

# 2. Load your fine-tuned checkpoint state dict
print("Loading fine-tuned weights...")
checkpoint = torch.load(CHECKPOINT_FILE, map_location="cpu")

# 3. Unwrap if needed (common keys: "model" or "module.")
state_dict = checkpoint.get("model", checkpoint)

# Remove "module." prefix if present (from DDP training)
state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

# 4. Inject the fine-tuned weights into the pipeline's instantiated model
pipeline.model.load_state_dict(state_dict, strict=False)
pipeline.model.eval()  # Ensure model is in inference mode
print("✅ Model weights loaded successfully!")

Device: cuda, Lang supported: True
Loading base model architecture...


/local/scratch/matuor/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/local/scratch/matuor/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/local/scratch/matuor/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Loading fine-tuned weights...
✅ Model weights loaded successfully!


In [ ]:
df_test = load_all_data("test")
df_test["sentence"] = df_test["sentence"].apply(normalize_romansh_text)
print(f"Loaded {len(df_test)} samples")

Loaded 631 samples


In [29]:
audio_paths = df_test["audio_path"].tolist()
transcriptions = []

for i in tqdm(range(0, len(audio_paths), BATCH_SIZE)):
    batch = audio_paths[i:i+BATCH_SIZE]
    try:
        # Transcribe the batch. (CTC models don't strictly enforce 'lang', 
        # but passing it ensures the pipeline is formatted correctly).
        results = pipeline.transcribe(
            batch, 
            lang=[LANGUAGE_CODE] * len(batch), 
            batch_size=len(batch)
        )
        transcriptions.extend(results)
    except Exception as e:
        print(f"Batch error at index {i}: {e}")
        transcriptions.extend([""] * len(batch))

df_test["omnilingual_transcription"] = transcriptions

 41%|████      | 32/79 [00:26<00:27,  1.70it/s]

Batch error at index 248: The map function has failed while processing the path 'data' of the input data. See nested exception for details.


100%|██████████| 79/79 [01:08<00:00,  1.15it/s]


In [30]:
# Cell 5: Compute metrics
df_test = add_metrics_columns(df_test, "sentence", "omnilingual_transcription")
summary = idiom_summary(df_test)
print_evaluation_summary(summary)

# Cell 6: Show examples
show_examples(df_test, "omnilingual_transcription")


OVERALL RESULTS
Total test samples: 631
Word Error Rate (WER): 26.59%
Character Error Rate (CER): 6.72%

PER IDIOM RESULTS

CC-2021-05-28
  Samples: 81
  WER: 10.47%
  CER: 2.42%

RMPUTER-CC-2021-06-11
  Samples: 114
  WER: 26.63%
  CER: 5.89%

RMSURMIRAN-CC-2021-12-23
  Samples: 151
  WER: 32.22%
  CER: 8.33%

RMSURSILV-CC-2021-05-28
  Samples: 94
  WER: 28.40%
  CER: 7.52%

RMSUTSILV-CC-2022-05-18
  Samples: 94
  WER: 32.93%
  CER: 8.85%

RMVALLADER-CC-2021-05-28
  Samples: 97
  WER: 23.80%
  CER: 6.06%

SUMMARY TABLE
                   idiom  samples  wer_mean  wer_std  cer_mean  cer_std
           cc-2021-05-28       81     10.47     7.46      2.42     2.04
   rmputer-cc-2021-06-11      114     26.63    23.66      5.89     4.85
rmsurmiran-cc-2021-12-23      151     32.22    20.20      8.33     6.12
 rmsursilv-cc-2021-05-28       94     28.40    12.81      7.52     4.41
 rmsutsilv-cc-2022-05-18       94     32.93    12.75      8.85     4.18
rmvallader-cc-2021-05-28       97     23.

In [10]:
import pyarrow.dataset as pa_ds
import polars as pl

ds = pa_ds.dataset(
    "../../parquet-dataset/rm-dataset/version=0",
    partitioning="hive",
)
table = ds.to_table(columns=["audio_size", "split"])
df = pl.from_arrow(table).filter(pl.col("split") == "train")

# Count how many samples are longer than 80_000 samples (5 seconds)
long_samples = df.filter(pl.col("audio_size") > 480_000)
print(f"Total train samples: {len(df)}")
print(f"Samples > 5s (80k): {len(long_samples)} ({len(long_samples)/len(df)*100:.1f}%)")

Total train samples: 29565
Samples > 5s (80k): 2145 (7.3%)
